Imports and configuration

In [7]:

from pathlib import Path
import gc
import json
import time

import chromadb
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from IPython.display import display

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]

    for candidate in candidates:
        expected_db = (
            candidate
            / "data"
            / "patent_subset_runs"
            / "20261022_190054"
            / "chroma"
        )

        if expected_db.is_dir():
            return candidate

    raise FileNotFoundError(
        f"Could not find the experiment database from {Path.cwd()}.\n"
        "Please check that the 20261022_190054/chroma directory exists."
    )

ROOT = find_project_root()
RUN_ID = "20261022_190054"
DB_ROOT = ROOT / "data" / "patent_subset_runs" / RUN_ID / "chroma"
OUTPUT_DIR = ROOT / "results" / "retrieval_experiment_01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Database root:", DB_ROOT)
print("Database root exists:", DB_ROOT.is_dir())
print("Output directory:", OUTPUT_DIR)


Project root: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project
Database root: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/data/patent_subset_runs/20261022_190054/chroma
Database root exists: True
Output directory: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/results/retrieval_experiment_01


Inspect the seven indexes

In [10]:

MODELS = {
    "patentsberta": {
        "model_id": "AI-Growth-Lab/PatentSBERTa",
        "collection": "patent_chunks_patentsberta",
        "max_length": 512,
        "query_prefix": "",
    },
    "paecter": {
        "model_id": "mpi-inno-comp/paecter",
        "collection": "patent_chunks_paecter",
        "max_length": 512,
        "query_prefix": "",
    },
    "nomic": {
        "model_id": "nomic-ai/nomic-embed-text-v1.5",
        "collection": "patent_chunks_nomic",
        "max_length": 8192,
        "query_prefix": "search_query: ",
    },
    "bge": {
        "model_id": "BAAI/bge-base-en-v1.5",
        "collection": "patent_chunks_bge",
        "max_length": 512,
        "query_prefix": "",
    },
    "e5": {
        "model_id": "intfloat/e5-base-v2",
        "collection": "patent_chunks_e5",
        "max_length": 512,
        "query_prefix": "query: ",
    },
    "mpnet": {
        "model_id": "sentence-transformers/all-mpnet-base-v2",
        "collection": "patent_chunks_mpnet",
        "max_length": 384,
        "query_prefix": "",
    },
    "minilm": {
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "collection": "patent_chunks_minilm",
        "max_length": 256,
        "query_prefix": "",
    },
}

print("Models configured:", len(MODELS))
print("Model keys:", list(MODELS.keys()))


Models configured: 7
Model keys: ['patentsberta', 'paecter', 'nomic', 'bge', 'e5', 'mpnet', 'minilm']


In [11]:
print("MODELS defined:", "MODELS" in globals())
print("DB_ROOT defined:", "DB_ROOT" in globals())
print("OUTPUT_DIR defined:", "OUTPUT_DIR" in globals())

MODELS defined: True
DB_ROOT defined: True
OUTPUT_DIR defined: True


In [12]:

index_rows = []
clients = {}

for model_key, config in MODELS.items():
    db_path = DB_ROOT / model_key

    row = {
        "model": model_key,
        "collection": config["collection"],
        "database_exists": db_path.is_dir(),
        "collection_exists": False,
        "count": None,
        "distance_metric": None,
        "status": "NOT CHECKED",
    }

    try:
        if not db_path.is_dir():
            row["status"] = "DATABASE MISSING"
            index_rows.append(row)
            continue

        client = chromadb.PersistentClient(path=str(db_path))

        collections = client.list_collections()
        collection_names = {
            c.name if hasattr(c, "name") else str(c)
            for c in collections
        }

        if config["collection"] not in collection_names:
            row["status"] = (
                "COLLECTION MISSING; found: "
                + ", ".join(sorted(collection_names))
            )
            index_rows.append(row)
            continue

        collection = client.get_collection(config["collection"])

        row["collection_exists"] = True
        row["count"] = collection.count()
        row["distance_metric"] = (
            (collection.metadata or {}).get("hnsw:space", "unspecified")
        )
        row["status"] = "OK"

        clients[model_key] = (client, collection)

    except Exception as exc:
        row["status"] = f"ERROR: {type(exc).__name__}: {exc}"

    index_rows.append(row)

index_df = pd.DataFrame(index_rows)

display(index_df)

index_df.to_csv(
    OUTPUT_DIR / "index_audit.csv",
    index=False
)

print("\nSaved audit to:", OUTPUT_DIR / "index_audit.csv")
print("\nAll seven indexes checked:", len(index_df) == 7)
print("All indexes OK:", (index_df["status"] == "OK").all())


,model,collection,database_exists,collection_exists,count,distance_metric,status
0,patentsberta,patent_chunks_patentsberta,True,True,28166,cosine,OK
1,paecter,patent_chunks_paecter,True,True,26548,cosine,OK
2,nomic,patent_chunks_nomic,True,True,5293,cosine,OK
3,bge,patent_chunks_bge,True,True,28166,cosine,OK
4,e5,patent_chunks_e5,True,True,28302,cosine,OK
5,mpnet,patent_chunks_mpnet,True,True,37195,cosine,OK
6,minilm,patent_chunks_minilm,True,True,57279,cosine,OK



Saved audit to: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/results/retrieval_experiment_01/index_audit.csv

All seven indexes checked: True
All indexes OK: True


Define representative test queries

In [13]:

TEST_QUERIES = {
    "motor_fault_detection": (
        "A system that detects faults in an electric motor using "
        "vibration measurements and signal analysis."
    ),
    "medical_image_analysis": (
        "A method for detecting abnormalities in medical images using "
        "machine learning."
    ),
    "wireless_communication": (
        "A wireless communication system that adapts transmission "
        "parameters according to channel conditions."
    ),
}

for query_name, query_text in TEST_QUERIES.items():
    print(f"\n{query_name}")
    print(query_text)



motor_fault_detection
A system that detects faults in an electric motor using vibration measurements and signal analysis.

medical_image_analysis
A method for detecting abnormalities in medical images using machine learning.

wireless_communication
A wireless communication system that adapts transmission parameters according to channel conditions.


Search one model at a time

In [14]:

TOP_K = 5
all_results = []

def search_one_model(model_key, query_name, query_text):
    config = MODELS[model_key]
    client, collection = clients[model_key]

    start = time.perf_counter()
    model = None

    try:
        model = SentenceTransformer(
            config["model_id"],
            device="cpu",
        )
        model.max_seq_length = config["max_length"]

        # Apply the model's query-side instruction where configured.
        prepared_query = config["query_prefix"] + query_text.strip()

        query_vector = model.encode(
            prepared_query,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        # ChromaDB already contains the document vectors.
        result = collection.query(
            query_embeddings=[np.asarray(query_vector, dtype=np.float32).tolist()],
            n_results=min(TOP_K, collection.count()),
            include=["documents", "metadatas", "distances"],
        )

        elapsed = time.perf_counter() - start
        rows = []

        ids = result["ids"][0]
        docs = result["documents"][0]
        metadatas = result["metadatas"][0]
        distances = result["distances"][0]

        for rank, (chunk_id, doc, meta, distance) in enumerate(
            zip(ids, docs, metadatas, distances), start=1
        ):
            meta = meta or {}

            rows.append({
                "query_name": query_name,
                "query": query_text,
                "model": model_key,
                "rank": rank,
                "patent_number": str(meta.get("patent_number", "")),
                "section": meta.get("section", ""),
                "cpc_label": meta.get("cpc_label", ""),
                "ipc_label": meta.get("ipc_label", ""),
                "decision": meta.get("decision", ""),
                "date_published": meta.get("date_published", ""),
                "chunk_id": chunk_id,
                "cosine_distance": float(distance),
                "similarity_1_minus_distance": 1.0 - float(distance),
                "elapsed_seconds": round(elapsed, 3),
                "document": doc or "",
            })

        return rows

    finally:
        if model is not None:
            del model
        gc.collect()


for query_name, query_text in TEST_QUERIES.items():
    print(f"\n{'=' * 65}")
    print("QUERY:", query_name)
    print(query_text)

    for model_key in MODELS:
        print(f"  Searching {model_key} ...", flush=True)

        try:
            rows = search_one_model(model_key, query_name, query_text)
            all_results.extend(rows)

            if rows:
                first = rows[0]
                print(
                    f"    Top result: patent {first['patent_number']} | "
                    f"distance={first['cosine_distance']:.4f} | "
                    f"{first['elapsed_seconds']:.2f}s"
                )
            else:
                print("    No results returned.")

        except Exception as exc:
            print(f"    ERROR: {type(exc).__name__}: {exc}")

results_df = pd.DataFrame(all_results)

if not results_df.empty:
    output_csv = OUTPUT_DIR / "smoke_test_results.csv"
    results_df.to_csv(output_csv, index=False)
    print("\nSaved results:", output_csv)
    print("Total retrieved chunks:", len(results_df))
else:
    print("\nNo results were collected. Inspect the errors above.")



QUERY: motor_fault_detection
A system that detects faults in an electric motor using vibration measurements and signal analysis.
  Searching patentsberta ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    Top result: patent 14902602 | distance=0.3607 | 15.02s
  Searching paecter ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

    Top result: patent 14902625 | distance=0.0721 | 20.49s
  Searching nomic ...


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

... (33 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    Top result: patent 14903033 | distance=0.3176 | 9.98s
  Searching minilm ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

    Top result: patent 14903033 | distance=0.3423 | 6.35s

Saved results: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/results/retrieval_experiment_01/smoke_test_results.csv
Total retrieved chunks: 105


Compare the results

In [15]:

if results_df.empty:
    print("No results available. Run Cell 4 successfully first.")
else:
    # Summary of unique patents retrieved by each model and query
    summary_df = (
        results_df.groupby(["query_name", "model"])
        .agg(
            chunks_retrieved=("chunk_id", "count"),
            unique_patents=("patent_number", "nunique"),
            mean_distance=("cosine_distance", "mean"),
            best_distance=("cosine_distance", "min"),
            elapsed_seconds=("elapsed_seconds", "first"),
        )
        .reset_index()
        .sort_values(["query_name", "model"])
    )

    display(summary_df)
    summary_df.to_csv(OUTPUT_DIR / "model_smoke_test_summary.csv", index=False)

    # Show the top five passages for a selected query
    selected_query = "motor_fault_detection"

    view_df = results_df[
        results_df["query_name"] == selected_query
    ][[
        "model",
        "rank",
        "patent_number",
        "section",
        "cosine_distance",
        "cpc_label",
        "document",
    ]].copy()

    view_df["document"] = view_df["document"].str.slice(0, 500)

    display(view_df.sort_values(["model", "rank"]))
    view_df.to_csv(
        OUTPUT_DIR / f"{selected_query}_top_results.csv",
        index=False,
    )

    print("\nFiles saved in:", OUTPUT_DIR)


,query_name,model,chunks_retrieved,unique_patents,mean_distance,best_distance,elapsed_seconds
0,medical_image_analysis,bge,5,4,0.344413,0.326210,11.396
1,medical_image_analysis,e5,5,3,0.158499,0.155421,10.424
2,medical_image_analysis,minilm,5,3,0.558119,0.534284,10.043
3,medical_image_analysis,mpnet,5,2,0.522343,0.490652,8.602
4,medical_image_analysis,nomic,5,4,0.261795,0.248040,10.137
5,medical_image_analysis,paecter,5,3,0.090061,0.087779,17.945
6,medical_image_analysis,patentsberta,5,5,0.419389,0.400899,15.416
7,motor_fault_detection,bge,5,5,0.307926,0.278919,11.237
8,motor_fault_detection,e5,5,5,0.157225,0.145166,11.684
9,motor_fault_detection,minilm,5,2,0.504960,0.486474,10.948


,model,rank,patent_number,section,cosine_distance,cpc_label,document
15,bge,1,14902554,title,0.278919,H02K338,ELECTRONICALLY COMMUTATED MOTOR
16,bge,2,14902625,background,0.294850,F01L1344,The published application asserts that this me...
17,bge,3,14890139,title,0.312823,H02K2336,BRUSH-COMMUNICATED DIRECT-CURRENT MOTOR
18,bge,4,14781851,description,0.325332,G01N27902,Since the balanced equilibrium of the bridge c...
19,bge,5,14902829,title,0.327706,G01F23284,FLUID GAUGE WITH LOW LEVEL DETECTION
20,e5,1,14902554,title,0.145166,H02K338,ELECTRONICALLY COMMUTATED MOTOR
21,e5,2,14902642,title,0.155379,H02K1102,ACTUATOR SYSTEM
22,e5,3,14902964,title,0.155662,G01L908,PIEZOELECTRIC SENSOR
23,e5,4,14902975,title,0.160031,E04B198,DAMPING DEVICE
24,e5,5,14902602,title,0.169886,F01L904,ELECTROMAGNETIC ACTUATING APPARATUS AND SYSTEM...



Files saved in: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/results/retrieval_experiment_01


Check for repeated patents

In [16]:

if not results_df.empty:
    duplicate_summary = (
        results_df.groupby(["query_name", "model"])
        .agg(
            returned_chunks=("chunk_id", "count"),
            unique_patents=("patent_number", "nunique"),
        )
        .reset_index()
    )

    duplicate_summary["extra_chunks_from_repeated_patents"] = (
        duplicate_summary["returned_chunks"]
        - duplicate_summary["unique_patents"]
    )

    display(duplicate_summary)


,query_name,model,returned_chunks,unique_patents,extra_chunks_from_repeated_patents
0,medical_image_analysis,bge,5,4,1
1,medical_image_analysis,e5,5,3,2
2,medical_image_analysis,minilm,5,3,2
3,medical_image_analysis,mpnet,5,2,3
4,medical_image_analysis,nomic,5,4,1
5,medical_image_analysis,paecter,5,3,2
6,medical_image_analysis,patentsberta,5,5,0
7,motor_fault_detection,bge,5,5,0
8,motor_fault_detection,e5,5,5,0
9,motor_fault_detection,minilm,5,2,3


In [17]:

# STEP 1: Audit metadata across all seven indexes
# Read-only: does not modify ChromaDB or regenerate embeddings.

from collections import Counter
import pandas as pd
from IPython.display import display

metadata_audit_rows = []
metadata_key_counts = {}
sample_metadata = {}

for model_key, config in MODELS.items():
    print(f"Inspecting: {model_key}")

    client, collection = clients[model_key]

    # Inspect a small sample only; do not load the entire collection.
    sample = collection.get(
        limit=5,
        include=["metadatas", "documents"]
    )

    metadatas = sample.get("metadatas") or []
    documents = sample.get("documents") or []

    # Which metadata keys are stored?
    key_counter = Counter()
    for metadata in metadatas:
        if metadata:
            key_counter.update(metadata.keys())

    metadata_key_counts[model_key] = dict(key_counter)

    # Preserve one example for manual inspection.
    sample_metadata[model_key] = (
        metadatas[0] if metadatas else {}
    )

    required_fields = [
        "patent_number",
        "chunk_index",
        "section",
        "title",
        "cpc_label",
        "ipc_label",
    ]

    row = {
        "model": model_key,
        "collection": config["collection"],
        "collection_count": collection.count(),
        "sampled_records": len(metadatas),
        "sample_documents_present": sum(
            bool(doc) for doc in documents
        ),
        "metadata_keys": ", ".join(sorted(key_counter.keys())),
    }

    for field in required_fields:
        row[f"has_{field}"] = (
            field in key_counter
        )

    row["sample_patent_numbers"] = ", ".join(
        str(m.get("patent_number", "MISSING"))
        for m in metadatas
    )

    metadata_audit_rows.append(row)

metadata_audit_df = pd.DataFrame(metadata_audit_rows)

display(metadata_audit_df)

print("\nExample metadata from each index:")
for model_key, metadata in sample_metadata.items():
    print(f"\n--- {model_key} ---")
    print(metadata)


Inspecting: patentsberta
Inspecting: paecter
Inspecting: nomic
Inspecting: bge
Inspecting: e5
Inspecting: mpnet
Inspecting: minilm


,model,collection,collection_count,sampled_records,sample_documents_present,metadata_keys,has_patent_number,has_chunk_index,has_section,has_title,has_cpc_label,has_ipc_label,sample_patent_numbers
0,patentsberta,patent_chunks_patentsberta,28166,5,5,"chunk_index, cpc_label, date_published, decisi...",True,True,True,False,True,True,"13261748, 13261748, 13261748, 13261748, 13261748"
1,paecter,patent_chunks_paecter,26548,5,5,"chunk_index, cpc_label, date_published, decisi...",True,True,True,False,True,True,"13261748, 13261748, 13261748, 13261748, 13261748"
2,nomic,patent_chunks_nomic,5293,5,5,"chunk_index, cpc_label, date_published, decisi...",True,True,True,False,True,True,"13261748, 13261748, 13261748, 13261748, 13261748"
3,bge,patent_chunks_bge,28166,5,5,"chunk_index, cpc_label, date_published, decisi...",True,True,True,False,True,True,"13261748, 13261748, 13261748, 13261748, 13261748"
4,e5,patent_chunks_e5,28302,5,5,"chunk_index, cpc_label, date_published, decisi...",True,True,True,False,True,True,"13261748, 13261748, 13261748, 13261748, 13261748"
5,mpnet,patent_chunks_mpnet,37195,5,5,"chunk_index, cpc_label, date_published, decisi...",True,True,True,False,True,True,"13261748, 13261748, 13261748, 13261748, 13261748"
6,minilm,patent_chunks_minilm,57279,5,5,"chunk_index, cpc_label, date_published, decisi...",True,True,True,False,True,True,"13261748, 13261748, 13261748, 13261748, 13261748"



Example metadata from each index:

--- patentsberta ---
{'source_row_index': 0, 'ipc_label': 'H04Q1100', 'patent_number': '13261748', 'token_count': 11, 'decision': 'ACCEPTED', 'source_split': 'train', 'date_published': '20160526', 'filing_date': '20160120', 'model_key': 'patentsberta', 'cpc_label': 'H04Q110071', 'model_id': 'AI-Growth-Lab/PatentSBERTa', 'patent_issue_date': '20170606', 'section': 'title', 'chunk_index': 0, 'document_prefix': ''}

--- paecter ---
{'ipc_label': 'H04Q1100', 'document_prefix': '', 'patent_number': '13261748', 'token_count': 11, 'source_split': 'train', 'model_id': 'mpi-inno-comp/paecter', 'model_key': 'paecter', 'chunk_index': 0, 'filing_date': '20160120', 'source_row_index': 0, 'patent_issue_date': '20170606', 'decision': 'ACCEPTED', 'date_published': '20160526', 'cpc_label': 'H04Q110071', 'section': 'title'}

--- nomic ---
{'ipc_label': 'H04Q1100', 'source_split': 'train', 'decision': 'ACCEPTED', 'model_id': 'nomic-ai/nomic-embed-text-v1.5', 'filing_da

In [18]:

# STEP 2: Audit patent IDs, chunk indices, and sections
# Read-only audit across all seven ChromaDB collections.

from collections import Counter
import pandas as pd
from IPython.display import display

metadata_integrity_rows = []

BATCH_SIZE = 5000

for model_key, config in MODELS.items():
    print(f"Auditing: {model_key}")

    _, collection = clients[model_key]

    total_records = collection.count()
    offset = 0

    missing_patent_id = 0
    empty_patent_id = 0
    missing_chunk_index = 0
    missing_section = 0

    patent_chunk_counts = Counter()
    chunk_identity_counts = Counter()
    section_counts = Counter()

    while offset < total_records:
        batch = collection.get(
            limit=BATCH_SIZE,
            offset=offset,
            include=["metadatas"]
        )

        metadatas = batch.get("metadatas") or []

        if not metadatas:
            break

        for metadata in metadatas:
            metadata = metadata or {}

            patent_id = metadata.get("patent_number")
            chunk_index = metadata.get("chunk_index")
            section = metadata.get("section")

            if patent_id is None:
                missing_patent_id += 1
            elif not str(patent_id).strip():
                empty_patent_id += 1
            else:
                patent_id = str(patent_id).strip()
                patent_chunk_counts[patent_id] += 1

                # Count repeated (patent, chunk index) combinations.
                if chunk_index is not None:
                    chunk_identity_counts[
                        (patent_id, str(chunk_index))
                    ] += 1

            if chunk_index is None:
                missing_chunk_index += 1

            if section is None or not str(section).strip():
                missing_section += 1
            else:
                section_counts[str(section)] += 1

        offset += len(metadatas)

    duplicate_chunk_pairs = sum(
        count - 1
        for count in chunk_identity_counts.values()
        if count > 1
    )

    metadata_integrity_rows.append({
        "model": model_key,
        "records": total_records,
        "unique_patents": len(patent_chunk_counts),
        "missing_patent_id": missing_patent_id,
        "empty_patent_id": empty_patent_id,
        "missing_chunk_index": missing_chunk_index,
        "missing_section": missing_section,
        "duplicate_patent_chunk_pairs": duplicate_chunk_pairs,
        "max_chunks_per_patent": (
            max(patent_chunk_counts.values())
            if patent_chunk_counts else 0
        ),
        "sections_found": ", ".join(sorted(section_counts)),
    })

metadata_integrity_df = pd.DataFrame(metadata_integrity_rows)

display(metadata_integrity_df)


Auditing: patentsberta
Auditing: paecter
Auditing: nomic
Auditing: bge
Auditing: e5
Auditing: mpnet
Auditing: minilm


,model,records,unique_patents,missing_patent_id,empty_patent_id,missing_chunk_index,missing_section,duplicate_patent_chunk_pairs,max_chunks_per_patent,sections_found
0,patentsberta,28166,600,0,0,0,0,6020,1007,"abstract, background, claims, description, sum..."
1,paecter,26548,600,0,0,0,0,5765,962,"abstract, background, claims, description, sum..."
2,nomic,5293,600,0,0,0,0,2792,106,"abstract, background, claims, description, sum..."
3,bge,28166,600,0,0,0,0,6020,1007,"abstract, background, claims, description, sum..."
4,e5,28302,600,0,0,0,0,6048,1013,"abstract, background, claims, description, sum..."
5,mpnet,37195,600,0,0,0,0,7427,1383,"abstract, background, claims, description, sum..."
6,minilm,57279,600,0,0,0,0,10728,2206,"abstract, background, claims, description, sum..."


In [20]:
# STEP 2: Locate the existing Streamlit apps and retrieval module
# Read-only: does not modify your project.

from pathlib import Path

ROOT = Path(
    "/home/patzer_adi/Documents/python/Python_Workspace/"
    "Patent_Intelligence_Project"
)

files_to_check = [
    "src/streamlit_app.py",
    "src/streamlit_graph_app.py",
    "src/streamlit_graph_app_neon.py",
    "src/rag_search.py",
]

for relative_path in files_to_check:
    path = ROOT / relative_path
    print(f"\n{'FOUND' if path.exists() else 'NOT FOUND'}: {relative_path}")

    if path.exists():
        print(f"Size: {path.stat().st_size:,} bytes")


FOUND: src/streamlit_app.py
Size: 8,099 bytes

FOUND: src/streamlit_graph_app.py
Size: 14,614 bytes

FOUND: src/streamlit_graph_app_neon.py
Size: 15,859 bytes

FOUND: src/rag_search.py
Size: 9,126 bytes


In [21]:
# STEP 3: Compare the three Streamlit apps

import ast

app_files = [
    ROOT / "src" / "streamlit_app.py",
    ROOT / "src" / "streamlit_graph_app.py",
    ROOT / "src" / "streamlit_graph_app_neon.py",
]

for path in app_files:
    print(f"\n{'=' * 70}")
    print(path.name)
    print("=" * 70)

    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    print("Functions:")
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print(f"  line {node.lineno}: {node.name}()")

    print("\nStreamlit components:")
    for line_no, line in enumerate(source.splitlines(), start=1):
        if "st." in line:
            print(f"  {line_no}: {line.strip()[:140]}")


streamlit_app.py
Functions:
  line 24: load_collection()
  line 29: load_embedding_model()
  line 34: load_reranker()
  line 38: _format_seconds()
  line 44: _chunk_label()

Streamlit components:
  18: st.set_page_config(page_title="Patent Intelligence", page_icon="🔎", layout="wide")
  19: st.title("Patent Intelligence")
  20: st.caption("Inspectable patent search over the existing ChromaDB index")
  23: @st.cache_resource(show_spinner="Opening the patent collection…")
  28: @st.cache_resource(show_spinner="Loading MiniLM query encoder…")
  33: @st.cache_resource(show_spinner="Loading cross-encoder reranker…")
  53: with st.sidebar:
  54: st.header("Search settings")
  55: db_path = st.text_input("ChromaDB directory", value=str(DEFAULT_DB_PATH))
  56: collection_name = st.text_input("Collection name", value=DEFAULT_COLLECTION_NAME)
  57: result_count = st.slider("Patent results", min_value=3, max_value=20, value=10)
  58: candidate_count = st.slider("Candidate chunks", min_value=10, m

In [22]:
# STEP 1 — Read-only project inventory
from pathlib import Path

ROOT = Path(
    "/home/patzer_adi/Documents/python/Python_Workspace/"
    "Patent_Intelligence_Project"
)

paths_to_check = [
    "src/streamlit_app.py",
    "src/streamlit_graph_app.py",
    "src/streamlit_graph_app_neon.py",
    "src/rag_search.py",
    "src/__init__.py",
    "data/sample_dataset",
    "data/chroma_db",
    "data/patent_subset_runs/20261022_190054/chroma",
    "data/patent_graph_2016/patent_similarity_graph.graphml",
    "data/patent_graph_2016/samples/patents_2016_sample_2500.parquet",
    "results",
    ".Patent_Intelligence_virtual_env_final/bin/python",
]

print("PROJECT ROOT:", ROOT)
print("ROOT EXISTS:", ROOT.exists())

for relative_path in paths_to_check:
    path = ROOT / relative_path
    status = "FOUND" if path.exists() else "MISSING"

    if path.is_dir():
        detail = "directory"
    elif path.is_file():
        detail = f"file ({path.stat().st_size:,} bytes)"
    else:
        detail = "not present"

    print(f"{status:8} | {relative_path} | {detail}")

PROJECT ROOT: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project
ROOT EXISTS: True
FOUND    | src/streamlit_app.py | file (8,099 bytes)
FOUND    | src/streamlit_graph_app.py | file (14,614 bytes)
FOUND    | src/streamlit_graph_app_neon.py | file (15,859 bytes)
FOUND    | src/rag_search.py | file (9,126 bytes)
FOUND    | src/__init__.py | file (60 bytes)
FOUND    | data/sample_dataset | directory
FOUND    | data/chroma_db | directory
FOUND    | data/patent_subset_runs/20261022_190054/chroma | directory
FOUND    | data/patent_graph_2016/patent_similarity_graph.graphml | file (3,113,878 bytes)
FOUND    | data/patent_graph_2016/samples/patents_2016_sample_2500.parquet | file (69,475,913 bytes)
FOUND    | results | directory
FOUND    | .Patent_Intelligence_virtual_env_final/bin/python | file (14,424 bytes)


In [23]:
# STEP 1 — Inspect the seven-model ChromaDB root
from pathlib import Path
import chromadb

ROOT = Path(
    "/home/patzer_adi/Documents/python/Python_Workspace/"
    "Patent_Intelligence_Project"
)

DB_ROOT = (
    ROOT
    / "data"
    / "patent_subset_runs"
    / "20261022_190054"
    / "chroma"
)

print("Database root:", DB_ROOT)
print("Exists:", DB_ROOT.exists())

if not DB_ROOT.exists():
    raise FileNotFoundError(DB_ROOT)

for db_path in sorted(p for p in DB_ROOT.iterdir() if p.is_dir()):
    try:
        client = chromadb.PersistentClient(path=str(db_path))
        collections = client.list_collections()

        print(f"\nDATABASE: {db_path.name}")
        if not collections:
            print("  No collections found.")

        for collection in collections:
            print(
                f"  Collection: {collection.name}"
                f" | Count: {collection.count():,}"
            )

    except Exception as exc:
        print(f"\nDATABASE: {db_path.name}")
        print(f"  ERROR: {type(exc).__name__}: {exc}")

Database root: /home/patzer_adi/Documents/python/Python_Workspace/Patent_Intelligence_Project/data/patent_subset_runs/20261022_190054/chroma
Exists: True

DATABASE: bge
  Collection: patent_chunks_bge | Count: 28,166

DATABASE: e5
  Collection: patent_chunks_e5 | Count: 28,302

DATABASE: minilm
  Collection: patent_chunks_minilm | Count: 57,279

DATABASE: mpnet
  Collection: patent_chunks_mpnet | Count: 37,195

DATABASE: nomic
  Collection: patent_chunks_nomic | Count: 5,293

DATABASE: paecter
  Collection: patent_chunks_paecter | Count: 26,548

DATABASE: patentsberta
  Collection: patent_chunks_patentsberta | Count: 28,166


In [24]:
from pathlib import Path
import json

ROOT = Path(
    "/home/patzer_adi/Documents/python/Python_Workspace/"
    "Patent_Intelligence_Project"
)

NOTEBOOK = ROOT / "notebooks" / "04_retrival_evaluation_model_comparision.ipynb"

assert ROOT.exists(), f"Project root not found: {ROOT}"
assert NOTEBOOK.exists(), f"Retrieval notebook not found: {NOTEBOOK}"

with NOTEBOOK.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Existing model configuration:")
found = False

for i, cell in enumerate(notebook.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if "MODELS" in source and ("prefix" in source.lower() or "collection" in source.lower()):
        print(f"\n--- Notebook cell {i} ---")
        print(source[:10000])
        found = True

if not found:
    print("No matching MODELS configuration cell found.")
    print("Notebook cells containing model/index configuration:")
    for i, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        if any(term in source.lower() for term in ("patentsberta", "paecter", "nomic", "collection_name")):
            print(f"\n--- Notebook cell {i} ---")
            print(source[:5000])

Existing model configuration:

--- Notebook cell 3 ---

MODELS = {
    "patentsberta": {
        "model_id": "AI-Growth-Lab/PatentSBERTa",
        "collection": "patent_chunks_patentsberta",
        "max_length": 512,
        "query_prefix": "",
    },
    "paecter": {
        "model_id": "mpi-inno-comp/paecter",
        "collection": "patent_chunks_paecter",
        "max_length": 512,
        "query_prefix": "",
    },
    "nomic": {
        "model_id": "nomic-ai/nomic-embed-text-v1.5",
        "collection": "patent_chunks_nomic",
        "max_length": 8192,
        "query_prefix": "search_query: ",
    },
    "bge": {
        "model_id": "BAAI/bge-base-en-v1.5",
        "collection": "patent_chunks_bge",
        "max_length": 512,
        "query_prefix": "",
    },
    "e5": {
        "model_id": "intfloat/e5-base-v2",
        "collection": "patent_chunks_e5",
        "max_length": 512,
        "query_prefix": "query: ",
    },
    "mpnet": {
        "model_id": "sentence-transformer